In [ ]:
import nltk
import random
import pandas as pd
 
from nltk.corpus import movie_reviews
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer ,WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
import re

In [ ]:
# nltk.download('movie_reviews')
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

In [ ]:
movie_reviews

In [ ]:
movie_reviews.categories()

In [ ]:
docs=[(movie_reviews.raw(fileid),category)
      for category in movie_reviews.categories()
      for fileid in movie_reviews.fileids(category)
      ]

In [ ]:
df=pd.DataFrame(docs, columns=["review","label"]).sample(200)
df["label"]=df["label"].map({"pos":1, "neg":0})

In [ ]:
df.head()

In [ ]:
df["label"].value_counts()

In [ ]:
import string

df["review"] = df["review"].str.lower()

translator = str.maketrans("", "", string.punctuation)

df["clean"] = df["review"].str.translate(translator)

In [ ]:
df["clean"].head()

In [ ]:
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [ ]:
stopwords=set(stopwords.words('english'))-{"not","no"," never"}

In [ ]:
stemmer=PorterStemmer()
def stem_pipeline(text):
    tokens=word_tokenize(text)
    filter=[w for w in tokens if w not in stopwords]
    stemmed=[stemmer.stem(w) for w in filter]
    return " ".join(stemmed)

In [ ]:
df["stemmed"]=df["clean"].apply(stem_pipeline)

In [ ]:
df["stemmed"].head() 

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemma_pipeline(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stopwords]
    pos_tags = pos_tag(filtered)

    lemmatized = [
        lemmatizer.lemmatize(w, get_wordnet_pos(tag))
        for w, tag in pos_tags
    ]

    return " ".join(lemmatized)

In [17]:
df["lemmatized"]=df["clean"].apply(lemma_pipeline)

In [18]:
df["lemmatized"].head()

261     summer movie season approach long awaited cess...
425     capsule hamhanded overunderwritten morality pl...
1901    start movie remind part movie stargate people ...
660     like get annoyed see people talk cellulars pub...
1244    john malkovich type film need see todays film ...
Name: lemmatized, dtype: object

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [20]:
x_stem=df["stemmed"]
x_lemma=df["lemmatized"]
y=df["label"]

In [21]:
x_train_s ,x_test_s ,y_train_s,y_test_s=train_test_split(x_stem,y,test_size=0.3,random_state=42)

x_train_l ,x_test_l ,y_train_l,y_test_l=train_test_split(x_lemma,y,test_size=0.3,random_state=42)

In [25]:
vectorizer_s = TfidfVectorizer()

x_train_s_vec = vectorizer_s.fit_transform(x_train_s)
x_test_s_vec = vectorizer_s.transform(x_test_s)


vectorizer_l = TfidfVectorizer()

x_train_l_vec = vectorizer_l.fit_transform(x_train_l)
x_test_l_vec = vectorizer_l.transform(x_test_l)

In [26]:
model=LogisticRegression(max_iter=1000)
model.fit(x_train_s_vec,y_train_s)

LogisticRegression(max_iter=1000)

In [28]:
y_pred_s=model.predict(x_test_s_vec)
acc_s=accuracy_score(y_test_s,y_pred_s)
acc_s

0.7333333333333333

In [29]:
model=LogisticRegression(max_iter=1000)
model.fit(x_train_l_vec,y_train_l)

LogisticRegression(max_iter=1000)

In [30]:
y_pred_l=model.predict(x_test_l_vec)
acc_l=accuracy_score(y_test_l,y_pred_l)
acc_l

0.6833333333333333

In [31]:
from sklearn.metrics import classification_report, confusion_matrix

print("STEMMING RESULTS")
print(classification_report(y_test_s, y_pred_s))

print("LEMMATIZATION RESULTS")
print(classification_report(y_test_l, y_pred_l))

STEMMING RESULTS
              precision    recall  f1-score   support

           0       0.67      0.90      0.76        29
           1       0.86      0.58      0.69        31

    accuracy                           0.73        60
   macro avg       0.76      0.74      0.73        60
weighted avg       0.77      0.73      0.73        60

LEMMATIZATION RESULTS
              precision    recall  f1-score   support

           0       0.63      0.83      0.72        29
           1       0.77      0.55      0.64        31

    accuracy                           0.68        60
   macro avg       0.70      0.69      0.68        60
weighted avg       0.70      0.68      0.68        60

